# SpaceX Launch Dashboard with Plotly Dash

Notebook implementation based on the provided IBM Plotly Dash lab. The dashboard uses the SpaceX launch dataset, a launch-site dropdown, a payload range slider, a success pie chart, and a payload-vs-outcome scatter chart.

**Source:** IBM lab PDF provided in this conversation. The lab specifies four tasks: launch-site dropdown, success pie-chart callback, payload range slider, and success-payload scatter-chart callback. fileciteturn0file0L2-L10

## 1. Install required packages

The original lab asks for `pandas` and `dash`. Plotly is used by Dash for the charts.

In [1]:
%pip install -q pandas dash plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 22.0 MB/s eta 0:00:00


## 2. Import libraries

In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
from IPython.display import display

## 3. Download and load the SpaceX dataset

The PDF specifies the dataset URL and the filename `spacex_launch_dash.csv`. fileciteturn0file0L42-L50

In [3]:
DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
df = pd.read_csv(DATA_URL)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

Rows: 56
Columns: 7


,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


## 4. Inspect the dataset

The lab refers to the launch-site field, `Payload Mass (kg)`, `class`, and `Booster Version Category`. fileciteturn0file0L180-L192

In [4]:
print("Column names:")
for col in df.columns:
    print(f"- {col}")

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

Column names:
- Unnamed: 0
- Flight Number
- Launch Site
- class
- Payload Mass (kg)
- Booster Version
- Booster Version Category

Data types:


,0
Unnamed: 0,int64
Flight Number,int64
Launch Site,object
class,int64
Payload Mass (kg),float64
Booster Version,object
Booster Version Category,object



Missing values:


,0
Unnamed: 0,0
Flight Number,0
Launch Site,0
class,0
Payload Mass (kg),0
Booster Version,0
Booster Version Category,0


## 5. Prepare variables for the dashboard

The PDF specifies a payload slider from 0 to 10,000 kg with a step of 1,000 kg, while its selected value should use the dataset's minimum and maximum payload values. fileciteturn0file0L154-L173

In [5]:
SITE_COL = "Launch Site"
PAYLOAD_COL = "Payload Mass (kg)"
CLASS_COL = "class"
BOOSTER_COL = "Booster Version Category"

# Validate that the fields required by the lab exist.
required_columns = [SITE_COL, PAYLOAD_COL, CLASS_COL, BOOSTER_COL]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise KeyError(f"Required columns not found: {missing_columns}")

launch_sites = sorted(df[SITE_COL].dropna().unique().tolist())
min_payload = float(df[PAYLOAD_COL].min())
max_payload = float(df[PAYLOAD_COL].max())

print("Launch sites:", launch_sites)
print(f"Payload range in dataset: {min_payload:,.0f} - {max_payload:,.0f} kg")

Launch sites: ['CCAFS LC-40', 'CCAFS SLC-40', 'KSC LC-39A', 'VAFB SLC-4E']
Payload range in dataset: 0 - 9,600 kg


## 6. Build the Dash application

The lab requires a dropdown with an `ALL` option, default value `ALL`, a placeholder, and search enabled. fileciteturn0file0L65-L80

In [6]:
app = Dash(__name__)

site_options = [
    {"label": "All Sites", "value": "ALL"}
] + [
    {"label": site, "value": site}
    for site in launch_sites
]

app.layout = html.Div([
    html.H1(
        "SpaceX Launch Records Dashboard",
        style={"textAlign": "center", "color": "#503D36", "fontSize": 40}
    ),

    html.Div([
        html.Label("Select a Launch Site:"),
        dcc.Dropdown(
            id="site-dropdown",
            options=site_options,
            value="ALL",
            placeholder="Select a Launch Site here",
            searchable=True
        )
    ], style={"width": "80%", "margin": "auto"}),

    html.Br(),

    html.Div([
        dcc.Graph(id="success-pie-chart")
    ]),

    html.Br(),

    html.Div([
        html.Label("Payload Range (kg):"),
        dcc.RangeSlider(
            id="payload-slider",
            min=0,
            max=10000,
            step=1000,
            value=[min_payload, max_payload],
            marks={i: str(i) for i in range(0, 10001, 1000)},
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={"width": "80%", "margin": "auto"}),

    html.Br(),

    html.Div([
        dcc.Graph(id="success-payload-scatter-chart")
    ])
], style={"padding": "20px"})

## 7. Callback for the success pie chart

The callback receives `site-dropdown` and returns the figure for `success-pie-chart`. For `ALL`, it shows total successful vs failed launches. For one site, it filters the dataframe first and then shows success/failure counts. This follows the logic specified in Task 2 of the lab. fileciteturn0file0L105-L120

In [7]:
@app.callback(
    Output(component_id="success-pie-chart", component_property="figure"),
    Input(component_id="site-dropdown", component_property="value")
)
def get_pie_chart(entered_site):
    if entered_site == "ALL":
        filtered_df = df
        title = "Total Launch Successes for All Sites"
    else:
        filtered_df = df[df[SITE_COL] == entered_site]
        title = f"Launch Successes for Site: {entered_site}"

    counts = (
        filtered_df[CLASS_COL]
        .value_counts()
        .reindex([1, 0], fill_value=0)
        .rename(index={1: "Success", 0: "Failure"})
        .reset_index()
    )
    counts.columns = ["Outcome", "Count"]

    fig = px.pie(
        counts,
        values="Count",
        names="Outcome",
        title=title
    )
    return fig

## 8. Callback for the payload scatter plot

Task 4 requires two inputs: the selected launch site and payload range. The scatter plot uses payload mass on the x-axis, launch outcome (`class`) on the y-axis, and booster version category for point coloring. fileciteturn0file0L178-L192

In [8]:
@app.callback(
    Output(component_id="success-payload-scatter-chart", component_property="figure"),
    [
        Input(component_id="site-dropdown", component_property="value"),
        Input(component_id="payload-slider", component_property="value")
    ]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range

    filtered_df = df[
        (df[PAYLOAD_COL] >= low) &
        (df[PAYLOAD_COL] <= high)
    ]

    if entered_site != "ALL":
        filtered_df = filtered_df[filtered_df[SITE_COL] == entered_site]

    title = (
        "Payload Mass vs. Launch Outcome"
        if entered_site == "ALL"
        else f"Payload Mass vs. Launch Outcome — {entered_site}"
    )

    fig = px.scatter(
        filtered_df,
        x=PAYLOAD_COL,
        y=CLASS_COL,
        color=BOOSTER_COL,
        hover_data=[SITE_COL, PAYLOAD_COL, CLASS_COL, BOOSTER_COL],
        title=title
    )

    fig.update_yaxes(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Failure", "Success"]
    )

    return fig

## 9. Quick local test of the callback logic

These cells do not start the web server. They simply verify that the callbacks return Plotly figures.

In [9]:
pie_test = get_pie_chart("ALL")
scatter_test = get_scatter_chart("ALL", [min_payload, max_payload])

print("Pie chart object:", type(pie_test).__name__)
print("Scatter chart object:", type(scatter_test).__name__)

Pie chart object: Figure
Scatter chart object: Figure


## 10. Run the dashboard

Run the next cell. Dash will start a local web server. In Jupyter, use the URL displayed by the cell to open the dashboard. The original lab uses port **8050**. fileciteturn0file0L52-L60

**Note:** Running this cell blocks the notebook kernel while the server is active. Stop/restart the kernel when you want to stop the server.

In [10]:
# Start the Dash server.
# If your environment already uses port 8050, change the port number.
app.run(debug=False, port=8050)

<IPython.core.display.Javascript object>

## 11. Answer the five visual-analysis questions with data

The lab asks these five questions after completing the dashboard: largest successful launches, highest site success rate, payload range with highest/lowest success rate, and booster version with highest success rate. fileciteturn0file0L196-L203

The following cells provide numerical checks to support what you observe in the dashboard.

### Question 1 — Which site has the largest successful launches?

In [11]:
successful_by_site = (
    df[df[CLASS_COL] == 1]
    .groupby(SITE_COL)
    .size()
    .sort_values(ascending=False)
)

display(successful_by_site.to_frame("Successful Launches"))
print("Site with the largest number of successful launches:", successful_by_site.idxmax())

,Successful Launches
Launch Site,
KSC LC-39A,10
CCAFS LC-40,7
VAFB SLC-4E,4
CCAFS SLC-40,3


Site with the largest number of successful launches: KSC LC-39A


### Question 2 — Which site has the highest launch success rate?

In [12]:
site_success_rate = (
    df.groupby(SITE_COL)[CLASS_COL]
    .agg(Launches="count", Successful="sum")
)
site_success_rate["Success Rate (%)"] = (
    site_success_rate["Successful"] / site_success_rate["Launches"] * 100
)

site_success_rate = site_success_rate.sort_values("Success Rate (%)", ascending=False)
display(site_success_rate)
print("Site with the highest success rate:", site_success_rate.index[0])

,Launches,Successful,Success Rate (%)
Launch Site,,,
KSC LC-39A,13,10,76.923077
CCAFS SLC-40,7,3,42.857143
VAFB SLC-4E,10,4,40.000000
CCAFS LC-40,26,7,26.923077


Site with the highest success rate: KSC LC-39A


### Questions 3 & 4 — Payload ranges with the highest and lowest success rates

The dashboard itself uses a continuous slider. For a numerical summary, this notebook groups payloads into 1,000-kg ranges, matching the slider's 1,000-kg step specified by the lab. fileciteturn0file0L157-L173

In [13]:
payload_df = df[[PAYLOAD_COL, CLASS_COL]].dropna().copy()

bins = list(range(0, 11001, 1000))
payload_df["Payload Range (kg)"] = pd.cut(
    payload_df[PAYLOAD_COL],
    bins=bins,
    right=False,
    include_lowest=True
)

payload_success_rate = (
    payload_df.groupby("Payload Range (kg)", observed=False)[CLASS_COL]
    .agg(Launches="count", Successful="sum")
)
payload_success_rate["Success Rate (%)"] = (
    payload_success_rate["Successful"] / payload_success_rate["Launches"] * 100
)
payload_success_rate = payload_success_rate[payload_success_rate["Launches"] > 0]

display(payload_success_rate)

max_rate = payload_success_rate["Success Rate (%)"].max()
min_rate = payload_success_rate["Success Rate (%)"].min()

print("Highest success-rate payload range(s):")
display(payload_success_rate[payload_success_rate["Success Rate (%)"] == max_rate])

print("Lowest success-rate payload range(s):")
display(payload_success_rate[payload_success_rate["Success Rate (%)"] == min_rate])

,Launches,Successful,Success Rate (%)
Payload Range (kg),,,
"[0, 1000)",10,2,20.000000
"[1000, 2000)",3,1,33.333333
"[2000, 3000)",10,5,50.000000
"[3000, 4000)",11,8,72.727273
"[4000, 5000)",8,3,37.500000
"[5000, 6000)",5,2,40.000000
"[6000, 7000)",4,0,0.000000
"[9000, 10000)",5,3,60.000000


Highest success-rate payload range(s):


,Launches,Successful,Success Rate (%)
Payload Range (kg),,,
"[3000, 4000)",11,8,72.727273


Lowest success-rate payload range(s):


,Launches,Successful,Success Rate (%)
Payload Range (kg),,,
"[6000, 7000)",4,0,0.0


### Question 5 — Which F9 Booster version has the highest launch success rate?

In [14]:
booster_success_rate = (
    df.groupby(BOOSTER_COL)[CLASS_COL]
    .agg(Launches="count", Successful="sum")
)
booster_success_rate["Success Rate (%)"] = (
    booster_success_rate["Successful"] / booster_success_rate["Launches"] * 100
)
booster_success_rate = booster_success_rate.sort_values("Success Rate (%)", ascending=False)

display(booster_success_rate)
print("Booster version with the highest success rate:", booster_success_rate.index[0])

,Launches,Successful,Success Rate (%)
Booster Version Category,,,
B5,1,1,100.000000
FT,24,16,66.666667
B4,11,6,54.545455
v1.1,15,1,6.666667
v1.0,5,0,0.000000


Booster version with the highest success rate: B5


## 12. Optional: save the processed dataset

This is optional and is not required by the lab. It can be useful if you want a local copy of the dataset used by the notebook.

In [15]:
df.to_csv("spacex_launch_dash.csv", index=False)
print("Saved: spacex_launch_dash.csv")

Saved: spacex_launch_dash.csv
